[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/GarretOS/python-ai-foundations/blob/main/projects/story-spark-api/story_spark.ipynb)

Open this notebook with the badge above, or upload a local copy to [Google Colab](https://colab.research.google.com/) using **File → Upload notebook**. The first setup cell downloads the required source files automatically in Colab.


# ✨ Story Spark API

This learning companion explains the existing `server.py` application and sends real HTTP requests to it. It does not maintain a second copy of the API. The API creates short story ideas, not full stories, and saves complete successful requests.

Run the cells in order in a fresh Python 3 runtime. The default run needs no Gemini key and makes no Gemini calls. Package installation needs internet access; optional generation also needs Gemini access and uses quota.


## 📁 Prepare the source files

In Google Colab, the next cell downloads **`server.py` and `requirements.txt`** from this project's published `main` branch into `/content`, then sets `PROJECT_DIR` to that folder. You do not need to upload source files. Downloads use Python's built-in `urllib` library, so this setup runs before package installation. An internet connection is required; a failed download reports the filename and URL so you can check access and rerun the cell.

Rerunning this setup in Colab refreshes those two source files from GitHub. Keep any source edits elsewhere before rerunning it. Credentials and history are not downloaded or overwritten.

Outside Colab, no downloads or file writes occur. For local Jupyter, open this notebook beside `server.py` and `requirements.txt`, or start Jupyter from the repository root. Use a Python 3.10+ kernel. Missing local files produce a clear error rather than being downloaded.


In [ ]:
from pathlib import Path

try:
    import google.colab
except ImportError:
    IN_COLAB = False
else:
    IN_COLAB = True

if IN_COLAB:
    from urllib.error import URLError
    from urllib.request import urlopen

    PROJECT_DIR = Path("/content")
    SOURCE_BASE_URL = (
        "https://raw.githubusercontent.com/GarretOS/python-ai-foundations/"
        "main/projects/story-spark-api/"
    )
    for filename in ("server.py", "requirements.txt"):
        url = SOURCE_BASE_URL + filename
        try:
            with urlopen(url, timeout=30) as response:
                source_bytes = response.read()
        except (URLError, OSError) as error:
            raise RuntimeError(
                f"Could not download {filename} from {url}. "
                "Check your internet connection and that the GitHub file is available, "
                "then rerun this setup cell."
            ) from error
        if not source_bytes:
            raise RuntimeError(f"Downloaded {filename} from {url} was empty. Rerun setup.")
        try:
            (PROJECT_DIR / filename).write_bytes(source_bytes)
        except OSError as error:
            raise RuntimeError(f"Could not save {filename} in /content. Check runtime storage.") from error
        print(f"Downloaded {filename} to {PROJECT_DIR}.")
else:
    PROJECT_DIR = Path.cwd()
    if not (PROJECT_DIR / "server.py").is_file():
        PROJECT_DIR = Path.cwd() / "projects" / "story-spark-api"
    if not all((PROJECT_DIR / name).is_file() for name in ("server.py", "requirements.txt")):
        raise FileNotFoundError(
            "Local source files are missing. Run Jupyter beside server.py and requirements.txt "
            "or from the repository root. No local files were downloaded or overwritten."
        )
    PROJECT_DIR = PROJECT_DIR.resolve()
    print("Using local source files:", PROJECT_DIR)


## 📦 Install dependencies

`fastapi` defines the routes, `uvicorn` serves the application, `google-genai` talks to Gemini, and `requests` is our HTTP client. `%pip` installs into the notebook kernel's Python environment using the project's actual dependency list. Installing packages does not call Gemini. If Colab asks for a session restart, restart and run from the top again.


In [ ]:
%pip install -q -r "{PROJECT_DIR / 'requirements.txt'}"


## 🔐 Configure the environment before starting the application

Keep `RUN_LIVE = False` for the offline lesson. No key is requested in that mode. For real generation, change it to `True` and rerun from this cell through startup and the demonstrations.

In Colab, add a secret named **GEMINI_API_KEY** in the Secrets panel (key icon) and enable notebook access. Outside Colab, an existing environment variable is used, or `getpass()` asks for a hidden value. Never put a key into a code cell. The key is passed to the child server through its environment, never command-line arguments or printed output.

The default model in `server.py` is `gemini-3.5-flash-lite`. `GEMINI_MODEL` overrides it; the cell preserves an existing override, or you can change `MODEL_NAME`. Configuration happens before the child process imports and initializes the application. After changing settings, rerun startup so the child receives the new environment.


In [ ]:
import os
from getpass import getpass

RUN_LIVE = False
MODEL_NAME = os.environ.get("GEMINI_MODEL", "gemini-3.5-flash-lite")
os.environ["GEMINI_MODEL"] = MODEL_NAME

if RUN_LIVE:
    try:
        from google.colab import userdata
    except ImportError:
        api_key = os.environ.get("GEMINI_API_KEY") or getpass("Gemini API key (hidden): ")
    else:
        # Colab raises a helpful error if the secret is missing or access is disabled.
        api_key = userdata.get("GEMINI_API_KEY")
    if not api_key or not api_key.strip():
        raise ValueError("Provide GEMINI_API_KEY before enabling live generation.")
    os.environ["GEMINI_API_KEY"] = api_key.strip()
    del api_key

print("Live generation enabled:", RUN_LIVE)
print("Model:", MODEL_NAME)


## 🧭 Client and server roles: GET and POST

The **server** listens for requests and decides what to return. The **client** sends requests. Here, Uvicorn runs FastAPI in a child Python process, and this notebook uses `requests` as the client. Both run inside the same Colab runtime, so the notebook can reach `127.0.0.1` without public hosting or a tunnel.

- `GET /stories` reads saved history and returns a JSON list, initially `[]`.
- `POST /generate-stories` sends a JSON object with story ingredients and asks the server to generate and save ideas. Success returns `{"stories": [...]}`.

An HTTP response contains a status code and a body. `200` means success; `400` and `422` indicate input problems. `response.json()` converts the JSON body into Python lists and dictionaries.


## 🐍 Read the existing application

We can inspect source text without importing the application into the notebook. The helper below uses Python's `ast` module to find a named function and prints its actual code, including any route decorators. These are teaching views of `server.py`, not new implementations.


In [ ]:
import ast

server_source = (PROJECT_DIR / "server.py").read_text(encoding="utf-8")
server_tree = ast.parse(server_source)

def show_function(name):
    for node in server_tree.body:
        if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)) and node.name == name:
            start = min([node.lineno] + [item.lineno for item in node.decorator_list])
            print("\n".join(server_source.splitlines()[start - 1:node.end_lineno]))
            return
    raise ValueError(f"Function not found: {name}")


## 📬 Decorators, Body parameters, and validation

`@app.post("/generate-stories")` is a **decorator**: it registers the function below it as the handler for that method and path. FastAPI reads the annotated `Body` parameters from the JSON request body.

`genre: str = Body(...)` requires a string. The `...` means the field is required. `num_stories: int = Body(3)` defaults to three when omitted. Missing required fields and unacceptable field types produce FastAPI validation errors (HTTP 422). FastAPI may convert some compatible values; this is not strict validation of every JSON type.

The handler strips surrounding whitespace, rejects blank text, and checks that the count is 1–3 (HTTP 400). These checks happen before the key check or any Gemini call. A valid request without a key returns HTTP 503.


In [ ]:
show_function("generate_stories")


## ⏳ async, await, and gather

An `async def` function creates a coroutine when called. `await` waits for an asynchronous operation and lets the event loop run other work while it waits. This helps with network requests; it does not make the model itself generate faster.

The handler creates one `generate_story(...)` coroutine per story. `asyncio.gather(..., return_exceptions=True)` runs them concurrently and waits for all results, preserving their order. If any result is an exception, the handler returns HTTP 502 and saves none of that request. Calls that already ran can still consume quota.

Notebooks already have an event loop. We avoid `asyncio.run()` in notebook cells by letting Uvicorn own its event loop in a separate process. Synchronous `requests` calls here talk to that independent server.


## 🧩 Gemini structured output and creative quality

`generate_story()` uses the asynchronous Gemini client. Its configuration requests JSON with a dictionary-based schema requiring `title`, `premise`, and `plot_twist` as strings. Python parses the JSON and `validate_story()` checks that all three values are nonblank strings. Keep both the schema and the extra validation: an earlier live attempt failed on an invalid `plot_twist`; adding the required response schema fixed the observed failure.

A valid structure does not guarantee creative quality, originality, or exact tone adherence. Ideas may be similar to one another even when each response passes validation. In the user's local example, two stories followed the cozy science-fiction ingredients but were not perfectly cozy. This notebook does not score creative quality.


In [ ]:
show_function("generate_story")
print()
show_function("validate_story")


## 💾 Storage and errors

`load_data()` returns `[]` only when the history file is missing. Unreadable JSON or a top-level value that is not a list raises an error; existing malformed data is preserved. It does not deeply validate every older entry.

Both routes share an `asyncio.Lock`. The generation route holds it while loading, appending, and saving, so concurrent requests in one process cannot overwrite each other's updates. `save_data()` writes a temporary file beside the history, flushes it, and uses `os.replace()` to replace the old file atomically. These synchronous file operations briefly occupy the event loop.

This is small local JSON storage, not a multi-process database. It has no pagination, size limit, or automatic backup. A read or save failure returns HTTP 500. If saving fails after generation, quota may have been used even though the request returns an error.


In [ ]:
show_function("load_data")
print()
show_function("save_data")


## 🖥️ Start a local notebook server

The next cell imports `server.py` only inside a child process and points its existing `DATA_FILE` variable at a separate notebook history file. It never uses the project's local `data.json`. The temporary history directory is reused for reruns in this kernel; a new kernel creates a new directory.

The launcher chooses an available local port, binds only to `127.0.0.1`, and waits up to 20 seconds for `/stories`. It stops the old notebook process before a rerun and cleans up if startup fails. The final cleanup cell stops the process too. Server logs are suppressed in this demonstration so provider diagnostics do not enter saved notebook outputs; startup failures are reported with the child exit code or timeout.

**Colab storage is temporary.** Download the notebook history from the Files sidebar if you want to keep live results before resetting or disconnecting the runtime. A local Jupyter temporary directory is also not a durable project history location.


In [ ]:
import atexit
import socket
import subprocess
import sys
import tempfile
import time

import requests

# On a cell rerun, the previous process is still available in notebook globals.
def stop_notebook_server():
    process = globals().get("notebook_server")
    if process is not None and process.poll() is None:
        process.terminate()
        try:
            process.wait(timeout=5)
        except subprocess.TimeoutExpired:
            process.kill()
            process.wait(timeout=5)

stop_notebook_server()
if "NOTEBOOK_HISTORY_DIR" not in globals():
    NOTEBOOK_HISTORY_DIR = Path(tempfile.mkdtemp(prefix="story-spark-notebook-"))
NOTEBOOK_HISTORY = NOTEBOOK_HISTORY_DIR / "history.json"

with socket.socket() as port_picker:
    port_picker.bind(("127.0.0.1", 0))
    notebook_port = port_picker.getsockname()[1]
BASE_URL = f"http://127.0.0.1:{notebook_port}"

server_environment = os.environ.copy()
server_environment["STORY_SPARK_NOTEBOOK_HISTORY"] = str(NOTEBOOK_HISTORY)
server_environment["STORY_SPARK_NOTEBOOK_PORT"] = str(notebook_port)
server_launcher = """
import os
from pathlib import Path
import server
import uvicorn

server.DATA_FILE = Path(os.environ["STORY_SPARK_NOTEBOOK_HISTORY"])
uvicorn.run(server.app, host="127.0.0.1",
            port=int(os.environ["STORY_SPARK_NOTEBOOK_PORT"]), log_level="warning")
"""
notebook_server = subprocess.Popen(
    [sys.executable, "-c", server_launcher],
    cwd=PROJECT_DIR,
    env=server_environment,
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)
# Popen has copied the environment; avoid retaining another copy of any key.
del server_environment
atexit.register(stop_notebook_server)

try:
    deadline = time.monotonic() + 20
    while time.monotonic() < deadline:
        if notebook_server.poll() is not None:
            raise RuntimeError(f"Notebook server exited with code {notebook_server.returncode}.")
        try:
            ready = requests.get(f"{BASE_URL}/stories", timeout=0.5)
        except requests.exceptions.RequestException:
            time.sleep(0.1)
            continue
        if ready.status_code != 200:
            raise RuntimeError(f"History readiness check returned HTTP {ready.status_code}.")
        break
    else:
        raise TimeoutError("Notebook server was not ready within 20 seconds.")
except BaseException:
    stop_notebook_server()
    raise

print("Notebook server ready:", BASE_URL)
print("Separate notebook history:", NOTEBOOK_HISTORY)


## 📖 GET: retrieve history

Our first actual HTTP request reads the notebook's saved requests. In a fresh runtime it returns an empty list. After successful live generation it includes the ingredients and stories. Reading history requires no Gemini key.


In [ ]:
response = requests.get(f"{BASE_URL}/stories", timeout=5)
assert response.status_code == 200, response.text
history_before_checks = response.json()
assert isinstance(history_before_checks, list)
print("HTTP", response.status_code)
print("Saved requests:", len(history_before_checks))
print(history_before_checks)


## 🧪 POST: reject invalid input without Gemini

These requests intentionally contain blank fields, an out-of-range count, or a missing required field. They run even when no key exists. We check the returned status and readable `detail` message, then confirm history is unchanged. No valid generation request is sent in this default flow.

The source shown earlier proves that input checks precede `gather()` and the provider call. A status-code test alone does not measure provider-call counts; implementation verification also uses a fail-fast stub to check that invalid requests never reach generation.


In [ ]:
valid_ingredients = {
    "genre": "cozy science fiction",
    "setting": "a floating night market on Europa",
    "character": "a retired cartographer",
    "num_stories": 2,
}
cases = [
    ("blank genre", {**valid_ingredients, "genre": "   "}, 400),
    ("blank setting", {**valid_ingredients, "setting": "\t"}, 400),
    ("blank character", {**valid_ingredients, "character": ""}, 400),
    ("count too low", {**valid_ingredients, "num_stories": 0}, 400),
    ("count too high", {**valid_ingredients, "num_stories": 4}, 400),
    ("missing genre", {key: value for key, value in valid_ingredients.items() if key != "genre"}, 422),
]
for label, payload, expected_status in cases:
    response = requests.post(f"{BASE_URL}/generate-stories", json=payload, timeout=5)
    assert response.status_code == expected_status, (label, response.status_code, response.text)
    print(f"{label}: HTTP {response.status_code} — {response.json()['detail']}")

response = requests.get(f"{BASE_URL}/stories", timeout=5)
assert response.status_code == 200, response.text
assert response.json() == history_before_checks
print("Invalid requests left history unchanged.")


## ✨ Optional: generate two real stories

This cell skips generation unless you explicitly enabled `RUN_LIVE` in the environment cell and restarted the notebook server. With live mode enabled, it sends the user's Europa example, checks all three nonblank fields for both stories, and prints readable results. A failed HTTP response raises an error rather than being presented as stories.

The longer HTTP timeout allows time for Gemini. A client timeout does not guarantee that server work was cancelled; check history before retrying to avoid duplicate saved requests. Model availability and quota depend on your Google account.


In [ ]:
if RUN_LIVE:
    response = requests.post(
        f"{BASE_URL}/generate-stories", json=valid_ingredients, timeout=120
    )
    if response.status_code != 200:
        raise RuntimeError(f"Generation returned HTTP {response.status_code}: {response.text}")
    stories = response.json()["stories"]
    assert isinstance(stories, list) and len(stories) == 2
    for number, story in enumerate(stories, start=1):
        for field in ("title", "premise", "plot_twist"):
            assert isinstance(story.get(field), str) and story[field].strip()
        print(f"\nStory {number}: {story['title']}")
        print("Premise:", story["premise"])
        print("Plot twist:", story["plot_twist"])

    response = requests.get(f"{BASE_URL}/stories", timeout=5)
    assert response.status_code == 200, response.text
    history = response.json()
    assert any(
        all(entry.get(key) == value for key, value in valid_ingredients.items())
        and entry.get("stories") == stories
        for entry in history
    )
    print(f"\nSaved and retrieved successfully ({len(history)} total requests).")
else:
    print("Live generation skipped. RUN_LIVE is False; no Gemini request was made.")


## 🧹 Cleanup and reruns

Stop the child server when finished. This leaves the separate history file available for download and does not delete project history. You can rerun startup and subsequent request cells to reuse the notebook history in this kernel. Rerunning startup while the server is running also stops the previous process first. Restarting the kernel gives the next run a new temporary history directory.


In [ ]:
stop_notebook_server()
assert notebook_server.poll() is not None
print("Notebook server stopped.")
print("History location (created only after successful generation):", NOTEBOOK_HISTORY)


## 📚 Review and verification scope

The local Python script remains the primary application. This notebook practices HTTP client/server roles, decorators, request bodies, validation, asynchronous network calls, structured model output, and JSON storage without duplicating the server.

The non-live notebook flow was executed in a fresh local Jupyter kernel, including startup, HTTP checks, reruns, and cleanup. No live Gemini call was made for this documentation task. The user's earlier local live checks confirmed two valid stories, saved-history retrieval, and persistence after restarting the server.

The user verified actual Google Colab execution: automatic source downloads and dependency installation succeeded; offline history retrieval and input-validation checks passed; invalid requests left history unchanged; live generation with `gemini-3.5-flash-lite` returned two stories, each with nonblank `title`, `premise`, and `plot_twist`; the generated request was saved and retrieved; and server cleanup succeeded.

For local two-terminal operation, offline `test.py` checks, `test.py --live`, and the full error behavior, see `README.md`.
